<a href="https://colab.research.google.com/github/Lariuki/Especializacao-Deep-Learning/blob/main/Dados_agregados.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import kagglehub
import os
import pandas as pd
import gc

In [2]:
# Download dados agregados
path = kagglehub.dataset_download("huseyincot/amex-agg-data-pickle")

print("Path to dataset files:", path)

100%|██████████| 2.76G/2.76G [00:20<00:00, 146MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/huseyincot/amex-agg-data-pickle/versions/4


In [3]:
os.listdir(path)

['test_agg.pkl', 'train_agg.pkl']

In [4]:
train_path = os.path.join(path, "train_agg.pkl")
test_path  = os.path.join(path, "test_agg.pkl")

train_df = pd.read_pickle(train_path, compression='gzip')
test_df  = pd.read_pickle(test_path, compression='gzip')

In [5]:
print(train_df.shape)
print(test_df.shape)

(458913, 919)
(924621, 918)


In [6]:
# verificando qual coluna tem a mais no train_df
set(train_df.columns) - set(test_df.columns)

{'target'}

Verificando colunas categoricas

In [7]:
# verificando se existe colunas categoricas
train_df.select_dtypes(include=['object', 'category']).columns

Index(['D_63_last', 'D_64_last'], dtype='object')

In [8]:
# verificando quantas categorias existem
train_df[['D_63_last', 'D_64_last']].nunique()

,0
D_63_last,6
D_64_last,4


In [9]:
from sklearn.preprocessing import OneHotEncoder

# Redefining the function locally to apply the fix for 'sparse' parameter
def one_hot_encode(cat_df, encoder=None):
    if encoder is None:
        # Corrected: changed sparse=False to sparse_output=False
        encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
        encoded = encoder.fit_transform(cat_df)
    else:
        encoded = encoder.transform(cat_df)

    encoded_df = pd.DataFrame(
        encoded,
        columns=encoder.get_feature_names_out(cat_df.columns),
        index=cat_df.index
    )
    return encoded_df, encoder

cat_cols = ['D_63_last', 'D_64_last']

train_cat = train_df[cat_cols]
train_cat_enc, encoder = one_hot_encode(train_cat)

In [10]:
print(train_cat.shape)
print(train_cat_enc.shape)

(458913, 2)
(458913, 11)


In [11]:
train_cat_enc.columns

Index(['D_63_last_CL', 'D_63_last_CO', 'D_63_last_CR', 'D_63_last_XL',
       'D_63_last_XM', 'D_63_last_XZ', 'D_64_last_-1', 'D_64_last_O',
       'D_64_last_R', 'D_64_last_U', 'D_64_last_nan'],
      dtype='object')

Verificando NaNs

In [12]:
# separando colunas categoricas de numericas
num_cols = train_df.select_dtypes(include=['float32','float64','float16','int']).columns
cat_cols = train_df.select_dtypes(include=['object','category']).columns

In [13]:
# verificando os nans por tipo de coluna
train_df[num_cols].isna().sum().sort_values(ascending=False).head(20)
train_df[cat_cols].isna().sum().sort_values(ascending=False).head(20)

,0
D_64_last,5279
D_63_last,0


In [14]:
cat_features = ['B_30', 'B_38', 'D_114', 'D_116', 'D_117', 'D_120', 'D_126', 'D_63', 'D_64', 'D_66', 'D_68']

In [15]:
[col for col in cat_features if col in train_df.columns]

[]

In [16]:
from typing import List
def impute_helper_aggregated(col, cat_cols=List):
    """
    Função segura para imputar valores faltantes no dataset agregado AMEX.

    - Preenche NaNs em colunas categóricas listadas em cat_cols com a moda
    - Preenche NaNs em colunas numéricas com a média
    - Ignora colunas one-hot (0/1)
    """
    convert_dtype = False

    # Converter float16 para float32 para calcular média com segurança
    if col.dtype == 'float16':
        convert_dtype = True
        col = col.astype('float32')

    # Se a coluna é categórica → preencher moda
    if col.name in cat_cols:
        col = col.fillna(col.mode()[0])
    # Se a coluna é numérica → preencher média
    elif col.dtype in ['float16','float32','float64','int']:
        col = col.fillna(col.mean())

    # Converter de volta para float16 se necessário
    if convert_dtype:
        col = col.astype('float16')

    return col



---



In [17]:
def impute_columns_aggregated(df, cat_cols=[]):
    """
    Imputa valores faltantes no dataset agregado.
    - Preenche NaNs em colunas categóricas (cat_cols) com moda
    - Preenche NaNs em colunas numéricas com média
    - Ignora colunas one-hot
    """
    # Aplicar função adaptada
    df = df.apply(impute_helper_aggregated, cat_cols=cat_cols)

    return df



---



In [18]:
import gc
def generate_x_y(df_file_path, test=False):

    df = pd.read_pickle(df_file_path, compression='gzip')
    y = None if test else df['target']

    # D_63_last and D_64_last columns are of type 'category', these are the only columns that need to be one-hot encoded
    # the other, original, categorical features are already modified from the aggregate functions
    encoded_df = one_hot_encode(df[['D_63_last', 'D_64_last']])

    # impute with numerical columns with mean() and categorical columns with most common value
    X = impute_columns_aggregated(df.drop(['D_63_last', 'D_64_last'], axis=1) if test else df.drop(['D_63_last', 'D_64_last', 'target'], axis=1))

    del df
    gc.collect()

    # combine new dataframes and sort them to line up when training/predicting
    X = pd.concat([X, encoded_df], axis=1)

    if test:
        return X
    else:
        return (X, y)

In [ ]:
X_train, y_train = generate_x_y(train_path)
X_train = X_train.reindex(sorted(X_train.columns), axis=1)

display(X_train.head())

X_train.to_pickle('X_train_agg.pkl', compression='gzip')
y_train.to_pickle('y_train_agg.pkl', compression='gzip')

del X_train, y_train
gc.collect()